# 03 Revenue Opportunity Forecasting

Purpose: build the 90-day forecast deliverable using the EDA-approved target. The EDA showed 29 monthly observations, 590,398 preserved source rows, and national observed enrollment growth of about 7.79%. Because the series is short, this notebook compares simple baselines against Prophet instead of assuming the most complex model is best.

Decision from EDA: model `observed_enrollment` first, and derive `proxy_revenue` with the documented PMPM assumption. Do not call this actual collections forecasting.

In [ ]:
from __future__ import annotations

import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, CPSC_DIR, PMPM_PROXY_REVENUE

for path in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

## Run Forecast Pipeline

Why: the reusable module trains and evaluates all models consistently. It writes the same outputs the dashboard and validation notebook consume.

In [ ]:
from src.models.forecast_prophet import run_forecasting

forecast_results, forecast_metrics = run_forecasting()
display(forecast_metrics)

## Model Selection Decision

Why: model selection must come from holdout performance, not preference. The last 3 months are used as the holdout window. Lower MAPE is better.

In [ ]:
best_models = forecast_metrics.sort_values(["target", "mape"]).groupby("target").head(1)
display(best_models)
print("Decision: use the best holdout-MAPE model for dashboard headline, while still showing all model metrics.")

## Forecast vs Actual: Holdout

Elements: x-axis is month; y-axis is target value; colored lines are models; black markers are actual holdout values.

Why: this shows whether the forecast method actually tracked the latest known months.

In [ ]:
for target in ["observed_enrollment", "proxy_revenue"]:
    holdout = forecast_results[(forecast_results["target"].eq(target)) & (forecast_results["period_type"].eq("holdout"))]
    fig = px.line(holdout, x="ds", y="yhat", color="model", markers=True, title=f"Holdout forecast comparison: {target}")
    actual = holdout.drop_duplicates("ds")[["ds", "actual"]]
    fig.add_scatter(x=actual["ds"], y=actual["actual"], mode="markers+lines", name="actual", marker=dict(color="black", size=10))
    fig.show()

## 90-Day Future Projection

Elements: x-axis is forecast month; y-axis is predicted observed enrollment or proxy revenue; color is model.

Why: the business deliverable asks for a 90-day projection. The MVP includes future predictions while keeping model comparison visible.

In [ ]:
for target in ["observed_enrollment", "proxy_revenue"]:
    future = forecast_results[(forecast_results["target"].eq(target)) & (forecast_results["period_type"].eq("future"))]
    fig = px.line(future, x="ds", y="yhat", color="model", markers=True, title=f"90-day future projection: {target}")
    fig.show()
    display(future.sort_values(["ds", "model"]))

## Forecasting Caveats

- `observed_enrollment` uses numeric CMS rows only; suppressed rows are preserved elsewhere and flagged.
- `proxy_revenue` equals observed enrollment multiplied by the documented PMPM proxy assumption.
- Prophet is included, but on this short series the simpler linear drift baseline performs best. That is an acceptable result; complexity does not win by default.